# **Convolución 2D**

Este cuaderno explora la operación de convolución 2D. Aquí escribirás la convolución a mano para comprobar que calculamos lo mismo que PyTorch.

Ejecuta las celdas en orden. En varios lugares verás la palabra "TAREA": sigue esas instrucciones, anticipa qué ocurrirá o escribe el código necesario para completar las funciones.

In [7]:
import numpy as np
import torch
# Configura una impresión legible
np.set_printoptions(precision=3, floatmode="fixed")
torch.set_printoptions(precision=3)

Esta rutina realiza una convolución en PyTorch

In [8]:
# Realiza la convolución en PyTorch
def conv_pytorch(image, conv_weights, stride=1, pad =1):
  # Convierte la imagen y el kernel en tensores
  image_tensor = torch.from_numpy(image) # (tamaño_lote, canales_entrada, alto_entrada, ancho_entrada)
  conv_weights_tensor = torch.from_numpy(conv_weights) # (canales_salida, canales_entrada, alto_kernel, ancho_kernel)
  # Ejecuta la convolución
  output_tensor = torch.nn.functional.conv2d(image_tensor, conv_weights_tensor, stride=stride, padding=pad)
  # Convierte de vuelta desde PyTorch y devuelve el resultado
  return(output_tensor.numpy()) # (tamaño_lote, canales_salida, alto_salida, ancho_salida)

Primero empezaremos con la convolución 2D más simple: un canal de entrada, un canal de salida y una sola imagen en el lote.

In [9]:
# Realiza la convolución en NumPy
def conv_numpy_1(image, weights, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + imageHeightIn - kernelHeight).astype(int)
    imageWidthOut = np.floor(1 + imageWidthIn - kernelWidth).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    # !!!!!! NOTA: HAY UN DETALLE IMPORTANTE AQUÍ !!!!!!!!
    # La imagen se rellenó con ceros, así que queda rodeada por un "anillo" de ceros
    # Eso significa que los índices de la imagen quedan desplazados por uno
    # Esto en realidad simplifica el código

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_kernel_y in range(kernelHeight):
          for c_kernel_x in range(kernelWidth):
            # TAREA -- recupera el píxel de la imagen y el peso de la convolución
            # Solo hay una imagen en el lote, un canal de entrada y un canal de salida, así que estos índices deben ser cero
            # Reemplaza las dos líneas siguientes
            this_pixel_value = 1.0
            this_weight = 1.0


            # Multiplica estos valores y súmalos a la salida en esta posición
            out[0, 0, c_y, c_x] += np.sum(this_pixel_value * this_weight)

    return out

In [10]:
import numpy as np

# Realiza la convolución en Numpy
def conv_numpy1(image, weights, pad=1):
    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0, 0), (pad, pad), (pad, pad)), 'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize, channels, imageHeight, imageWidth = image.shape
    kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = imageHeight - kernelHeight + 1
    imageWidthOut = imageWidth - kernelWidth + 1

    # Crea la salida
    out = np.zeros((batchSize, channels, imageHeightOut, imageWidthOut), dtype=np.float32)

    # Recorre cada posición de la imagen
    for n in range(batchSize):
        for c in range(channels):
            for i in range(imageHeightOut):
                for j in range(imageWidthOut):
                    # Extrae la región de la imagen que corresponde al kernel
                    region = image[n, c, i:i+kernelHeight, j:j+kernelWidth]

                    # Multiplica elemento a elemento y suma
                    out[n, c, i, j] = np.sum(region * weights)

    return out


In [11]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 1
image_height = 4
image_width = 6
channels_in = 1
kernel_size = 3
channels_out = 1

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_1(input_image, conv_weights)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

Resultados de PyTorch
[[[[-0.929 -2.760  0.716  0.114  0.560 -0.387]
   [-1.515  0.283  1.008  0.466 -1.094  2.004]
   [-1.634  3.555 -2.154 -0.892 -1.856  2.299]
   [ 0.565 -0.947 -0.629  2.996 -1.811 -0.533]]]]
Tus resultados
[[[[9.000 9.000 9.000 9.000 9.000 9.000]
   [9.000 9.000 9.000 9.000 9.000 9.000]
   [9.000 9.000 9.000 9.000 9.000 9.000]
   [9.000 9.000 9.000 9.000 9.000 9.000]]]]
Mean Absolute Error PyTorch y NumPy
9.107269073322657


Ahora añadamos la posibilidad de usar distintos pasos

In [12]:
# Realiza la convolución en NumPy
def conv_numpy_2(image, weights, stride=1, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_kernel_y in range(kernelHeight):
          for c_kernel_x in range(kernelWidth):
            # TAREA -- recupera el píxel de la imagen y el peso de la convolución
            # Solo hay una imagen en el lote, un canal de entrada y un canal de salida, así que estos índices deben ser cero
            # Reemplaza las dos líneas siguientes
            this_pixel_value = 1.0
            this_weight = 1.0


            # Multiplica estos valores y súmalos a la salida en esta posición
            out[0, 0, c_y, c_x] += np.sum(this_pixel_value * this_weight)

    return out

In [13]:
import numpy as np

def conv_numpy_1(image, weights, pad=1):
    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0,0),(0,0),(pad,pad),(pad,pad)), 'constant')

    # Formas
    batchSize, channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn_w, kernelHeight, kernelWidth = weights.shape

    # Salida
    imageHeightOut = int(np.floor(1 + imageHeightIn - kernelHeight))
    imageWidthOut  = int(np.floor(1 + imageWidthIn  - kernelWidth))
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    # Solo hay 1 imagen en el lote y 1 canal en este ejercicio, pero escribimos el bucle general
    for c_y in range(imageHeightOut):
        for c_x in range(imageWidthOut):
            for k_y in range(kernelHeight):
                for k_x in range(kernelWidth):
                    # Recupera el píxel (batch 0, canal entrada 0) y el peso (salida 0, entrada 0)
                    this_pixel_value = image[0, 0, c_y + k_y, c_x + k_x]
                    this_weight = weights[0, 0, k_y, k_x]

                    # Acumula
                    out[0, 0, c_y, c_x] += this_pixel_value * this_weight

    return out


In [14]:
def conv_numpy_2(image, weights, stride=1, pad=1):
    if pad != 0:
        image = np.pad(image, ((0,0),(0,0),(pad,pad),(pad,pad)), 'constant')

    batchSize, channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn_w, kernelHeight, kernelWidth = weights.shape

    imageHeightOut = int(np.floor(1 + (imageHeightIn - kernelHeight) / stride))
    imageWidthOut  = int(np.floor(1 + (imageWidthIn  - kernelWidth)  / stride))
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    # En el ejemplo del cuaderno hay 1 imagen en el lote, 1 canal in/out, pero implementamos la lógica general
    for c_y in range(imageHeightOut):
        for c_x in range(imageWidthOut):
            for c_out in range(channelsOut):
                for c_in in range(channelsIn):
                    for k_y in range(kernelHeight):
                        for k_x in range(kernelWidth):
                            in_y = c_y * stride + k_y
                            in_x = c_x * stride + k_x
                            this_pixel_value = image[0, c_in, in_y, in_x]
                            this_weight = weights[c_out, c_in, k_y, k_x]
                            out[0, c_out, c_y, c_x] += this_pixel_value * this_weight

    return out


In [15]:
def conv_numpy_3(image, weights, stride=1, pad=1):
    if pad != 0:
        image = np.pad(image, ((0,0),(0,0),(pad,pad),(pad,pad)), 'constant')

    batchSize, channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn_w, kernelHeight, kernelWidth = weights.shape

    imageHeightOut = int(np.floor(1 + (imageHeightIn - kernelHeight) / stride))
    imageWidthOut  = int(np.floor(1 + (imageWidthIn  - kernelWidth)  / stride))
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    # Recorremos canales de salida y entrada
    for c_y in range(imageHeightOut):
        for c_x in range(imageWidthOut):
            for c_out in range(channelsOut):
                for c_in in range(channelsIn):
                    for k_y in range(kernelHeight):
                        for k_x in range(kernelWidth):
                            in_y = c_y * stride + k_y
                            in_x = c_x * stride + k_x
                            # batch 0 en el ejemplo del cuaderno
                            this_pixel_value = image[0, c_in, in_y, in_x]
                            this_weight = weights[c_out, c_in, k_y, k_x]
                            out[0, c_out, c_y, c_x] += this_pixel_value * this_weight

    return out


In [16]:
def conv_numpy_4(image, weights, stride=1, pad=1):
    if pad != 0:
        image = np.pad(image, ((0,0),(0,0),(pad,pad),(pad,pad)), 'constant')

    batchSize, channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn_w, kernelHeight, kernelWidth = weights.shape

    imageHeightOut = int(np.floor(1 + (imageHeightIn - kernelHeight) / stride))
    imageWidthOut  = int(np.floor(1 + (imageWidthIn  - kernelWidth)  / stride))
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_batch in range(batchSize):
        for c_y in range(imageHeightOut):
            for c_x in range(imageWidthOut):
                for c_out in range(channelsOut):
                    for c_in in range(channelsIn):
                        for k_y in range(kernelHeight):
                            for k_x in range(kernelWidth):
                                in_y = c_y * stride + k_y
                                in_x = c_x * stride + k_x
                                this_pixel_value = image[c_batch, c_in, in_y, in_x]
                                this_weight = weights[c_out, c_in, k_y, k_x]
                                out[c_batch, c_out, c_y, c_x] += this_pixel_value * this_weight

    return out


In [17]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 1
image_height = 12
image_width = 10
channels_in = 1
kernel_size = 3
channels_out = 1
stride = 2

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_2(input_image, conv_weights, stride, pad=1)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

Resultados de PyTorch
[[[[-0.809 -4.550 -5.486 -9.506 -4.512]
   [-0.055  1.145 -5.388 -3.910  0.097]
   [-0.186  0.660  1.630  2.275  4.874]
   [ 2.386 -0.225  3.288 -4.239 -1.403]
   [ 0.825  1.710 -3.246  3.246  1.709]
   [ 0.809  3.695  3.491 -2.113 -2.714]]]]
Tus resultados
[[[[-0.809 -4.550 -5.486 -9.506 -4.512]
   [-0.055  1.145 -5.388 -3.910  0.097]
   [-0.186  0.660  1.630  2.275  4.874]
   [ 2.386 -0.225  3.288 -4.239 -1.403]
   [ 0.825  1.710 -3.246  3.246  1.709]
   [ 0.809  3.695  3.491 -2.113 -2.714]]]]
Mean Absolute Error PyTorch y NumPy
1.4221332014878001e-07


Ahora introduciremos varios canales de entrada y de salida

In [18]:
# Realiza la convolución en NumPy
def conv_numpy_3(image, weights, stride=1, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_y in range(imageHeightOut):
      for c_x in range(imageWidthOut):
        for c_channel_out in range(channelsOut):
          for c_channel_in in range(channelsIn):
            for c_kernel_y in range(kernelHeight):
              for c_kernel_x in range(kernelWidth):
                  # TAREA -- recupera el píxel de la imagen y el peso de la convolución
                  # Solo hay una imagen en el lote, así que este índice debe ser cero
                  # Reemplaza las dos líneas siguientes
                  this_pixel_value = 1.0
                  this_weight = 1.0

                  # Multiplica estos valores y súmalos a la salida en esta posición
                  out[0, c_channel_out, c_y, c_x] += np.sum(this_pixel_value * this_weight)
    return out

In [19]:
import numpy as np

def conv_numpy_2Image(image, weights, stride=1, pad=1):
    # Aplica relleno de ceros (batch, channels, H, W)
    if pad != 0:
        image = np.pad(image, ((0, 0), (0, 0), (pad, pad), (pad, pad)), 'constant')

    # Formas
    batchSize, channelIn, imageHeight, imageWidth = image.shape
    channelOut, channelIn_w, kernelHeight, kernelWidth = weights.shape

    # Tamaño de salida (la imagen ya incluye el padding)
    imageHeightOut = int(np.floor((imageHeight - kernelHeight) / stride) + 1)
    imageWidthOut  = int(np.floor((imageWidth  - kernelWidth)  / stride) + 1)

    # Salida
    out = np.zeros((batchSize, channelOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    # Convolución: recorremos batch, posiciones de salida, canales out/in y kernel
    for n in range(batchSize):
        for c_y in range(imageHeightOut):
            for c_x in range(imageWidthOut):
                for c_out in range(channelOut):
                    for c_in in range(channelIn):
                        for k_y in range(kernelHeight):
                            for k_x in range(kernelWidth):
                                in_y = c_y * stride + k_y
                                in_x = c_x * stride + k_x
                                this_pixel_value = image[n, c_in, in_y, in_x]
                                this_weight = weights[c_out, c_in, k_y, k_x]
                                out[n, c_out, c_y, c_x] += this_pixel_value * this_weight

    return out


In [20]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 1
image_height = 4
image_width = 6
channels_in = 5
kernel_size = 3
channels_out = 2

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_3(input_image, conv_weights, stride=1, pad=1)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

Resultados de PyTorch
[[[[ -0.785   5.463  -2.480   5.026  -3.594   7.785]
   [ -6.744   2.534  -0.664   7.149  -9.839   7.849]
   [ -4.794  14.074  -1.060   2.706 -10.182   2.004]
   [  1.809   0.287   4.648  -1.840   3.259   1.073]]

  [[  4.150   5.372   1.699   0.500   0.589   4.361]
   [ -4.123   5.136   4.677  -3.895  -4.990   2.546]
   [  3.991   5.768  -2.315   8.473   1.752   2.766]
   [  1.529   0.318  11.518  -5.444  -2.293   1.270]]]]
Tus resultados
[[[[45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000 45.000 45.000 45.000 45.000 45.000]]

  [[45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000 45.000 45.000 45.000 45.000 45.000]]]]
Mean Absolute Error PyTorch y NumPy
43.60341132892643


Ahora haremos la convolución completa con varias imágenes (tamaño de lote > 1), varios canales de entrada y varios canales de salida.

In [21]:
# Realiza la convolución en NumPy
def conv_numpy_4(image, weights, stride=1, pad=1):

    # Aplica relleno de ceros
    if pad != 0:
        image = np.pad(image, ((0, 0), (0 ,0), (pad, pad), (pad, pad)),'constant')

    # Obtiene los tamaños del arreglo de imagen y de los pesos del kernel
    batchSize,  channelsIn, imageHeightIn, imageWidthIn = image.shape
    channelsOut, channelsIn, kernelHeight, kernelWidth = weights.shape

    # Obtiene el tamaño de los arreglos de salida
    imageHeightOut = np.floor(1 + (imageHeightIn - kernelHeight) / stride).astype(int)
    imageWidthOut = np.floor(1 + (imageWidthIn - kernelWidth) / stride).astype(int)

    # Crea la salida
    out = np.zeros((batchSize, channelsOut, imageHeightOut, imageWidthOut), dtype=np.float32)

    for c_batch in range(batchSize):
      for c_y in range(imageHeightOut):
        for c_x in range(imageWidthOut):
          for c_channel_out in range(channelsOut):
            for c_channel_in in range(channelsIn):
              for c_kernel_y in range(kernelHeight):
                for c_kernel_x in range(kernelWidth):
                    # TAREA -- recupera el píxel de la imagen y el peso de la convolución
                    # Reemplaza las dos líneas siguientes
                    this_pixel_value = 1.0
                    this_weight = 1.0



                    # Multiplica estos valores y súmalos a la salida en esta posición
                    out[c_batch, c_channel_out, c_y, c_x] += np.sum(this_pixel_value * this_weight)
    return out

In [22]:
# Fija la semilla aleatoria para obtener siempre la misma respuesta
np.random.seed(1)
n_batch = 2
image_height = 4
image_width = 6
channels_in = 5
kernel_size = 3
channels_out = 2

# Crea una imagen de entrada aleatoria
input_image= np.random.normal(size=(n_batch, channels_in, image_height, image_width))
# Crea pesos aleatorios para el kernel de convolución
conv_weights = np.random.normal(size=(channels_out, channels_in, kernel_size, kernel_size))

# Realiza la convolución usando PyTorch
conv_results_pytorch = conv_pytorch(input_image, conv_weights, stride=1, pad=1)
print("Resultados de PyTorch")
print(conv_results_pytorch)

# Realiza la convolución en NumPy
print("Tus resultados")
conv_results_numpy = conv_numpy_4(input_image, conv_weights, stride=1, pad=1)
print(conv_results_numpy)

print("Mean Absolute Error PyTorch y NumPy")
print(np.mean(np.abs(conv_results_pytorch - conv_results_numpy)))

Resultados de PyTorch
[[[[ -3.633  -1.644   0.169  -1.167  -3.865   6.045]
   [ -9.004   7.303   4.414   0.361  -6.739   3.939]
   [ -1.391  13.502   3.807  -9.379   3.991   5.442]
   [  2.805   6.874  -9.287  -4.468  -1.501   4.607]]

  [[  1.940  -1.410   2.397  -0.235  -0.394  -1.483]
   [  5.049  -3.335  -7.596  -1.586   3.049  -1.857]
   [  3.514   0.475  -1.952  -1.291  -0.589  -0.948]
   [  6.524  -0.020  -3.298  -1.248   3.249  -2.680]]]


 [[[  4.154  -4.764  11.635   0.506  -4.012  -2.081]
   [ -1.125  -0.677  16.749  -7.030  -5.978  -2.629]
   [  0.778  -3.984 -10.284   1.575  -8.888   1.163]
   [  0.556  -2.290   1.407  -3.088   2.227  -5.403]]

  [[  1.048   4.322  -0.901  -5.820   3.998   2.281]
   [ -1.313   8.409  -0.100  14.625   1.223  -3.572]
   [  1.411   1.617  -4.078  -8.107   3.705   0.229]
   [ -3.540  -5.292  -5.619  -4.039  -4.048  -3.446]]]]
Tus resultados
[[[[45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000 45.000 45.000 45.000 45.000 45.000]
   [45.000

## Paralelismo en GPU para convoluciones 2D

Hasta ahora implementamos la convolución con ciclos anidados para entender qué operación se está calculando. Ese enfoque es útil para aprender, pero no es eficiente: cada salida se calcula una por una en Python.

En una CNN real, una convolución multicanal produce un tensor de salida con muchas posiciones independientes. Para cada imagen del lote, cada canal de salida y cada posición espacial se calcula:

$$
y_{n,c_o,i,j} = \sum_{c_i=0}^{C_i-1}\sum_{u=0}^{K_h-1}\sum_{v=0}^{K_w-1}
x_{n,c_i,i+u,j+v}\, w_{c_o,c_i,u,v}
$$

El valor $y_{n,c_o,i,j}$ no necesita esperar a que se calcule $y_{n,c_o,i,j+1}$ ni el de otro canal de salida. Por eso el trabajo se puede repartir en paralelo sobre varias dimensiones:

- distintas imágenes del lote `n`,
- distintos canales de salida `c_o`,
- distintas posiciones espaciales `(i, j)`.

CUDA aprovecha esta independencia asignando muchos de esos cálculos a miles de hilos de la GPU. Cada hilo o grupo de hilos calcula una parte del tensor de salida, mientras la suma sobre canales de entrada y posiciones del kernel se realiza con operaciones optimizadas. Por eso una convolución grande, con muchos canales y muchas posiciones espaciales, suele ser mucho más rápida en GPU que en CPU.

En PyTorch no escribimos los hilos CUDA manualmente. Movemos los tensores al dispositivo adecuado con `.cuda()` y usamos la misma función `torch.nn.functional.conv2d`. Si no hay CUDA, la celda también revisa `mps`, que es el backend de Metal usado en Apple Silicon.

La siguiente comparación no busca entrenar una red; solo mide cuánto tarda una convolución 2D grande en CPU y en GPU cuando el hardware está disponible.


In [23]:
# Comparación de una convolución 2D grande en CPU, CUDA y MPS si están disponibles
import time
import torch
import torch.nn.functional as F


def sincronizar(device):
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps" and hasattr(torch, "mps"):
        torch.mps.synchronize()


def medir_conv2d(x, w, device, repeticiones=3, calentamiento=1):
    x_dev = x.to(device)
    w_dev = w.to(device)

    with torch.no_grad():
        for _ in range(calentamiento):
            y = F.conv2d(x_dev, w_dev, stride=1, padding=1)
        sincronizar(device)

        inicio = time.perf_counter()
        for _ in range(repeticiones):
            y = F.conv2d(x_dev, w_dev, stride=1, padding=1)
        sincronizar(device)
        tiempo_promedio = (time.perf_counter() - inicio) / repeticiones

    return tiempo_promedio, y


# Tamaño grande pero ajustable. Si la CPU tarda demasiado, reduce image_size o los canales.
torch.manual_seed(0)
n_batch = 8
channels_in = 32
channels_out = 64
image_size = 128
kernel_size = 3

x = torch.randn(n_batch, channels_in, image_size, image_size)
w = torch.randn(channels_out, channels_in, kernel_size, kernel_size)

print(f"Entrada: {tuple(x.shape)}")
print("Formato: (n_batch, channels, height, width)")
print(f"Kernel:  {tuple(w.shape)}")
print("Formato del kernel: (channels_out, channels_in, kernel_height, kernel_width)")
print(f"Salida esperada: ({n_batch}, {channels_out}, {image_size}, {image_size})")

cpu = torch.device("cpu")
x_cpu = x.cpu()
w_cpu = w.cpu()
tiempo_cpu, y_cpu = medir_conv2d(x_cpu, w_cpu, cpu, repeticiones=1, calentamiento=1)
print(f"CPU:  {tiempo_cpu:.4f} segundos por convolución")

if torch.cuda.is_available():
    cuda = torch.device("cuda")
    x_cuda = x.cuda()
    w_cuda = w.cuda()
    tiempo_cuda, y_cuda = medir_conv2d(x_cuda, w_cuda, cuda, repeticiones=5, calentamiento=2)
    diferencia = (y_cpu - y_cuda.cpu()).abs().max().item()
    print(f"CUDA: {tiempo_cuda:.4f} segundos por convolución")
    print(f"Aceleración CPU/CUDA: {tiempo_cpu / tiempo_cuda:.2f}x")
    print(f"Diferencia máxima CPU vs CUDA: {diferencia:.6f}")
else:
    print("CUDA no está disponible en este equipo.")

    mps_disponible = hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    if mps_disponible:
        mps = torch.device("mps")
        tiempo_mps, y_mps = medir_conv2d(x, w, mps, repeticiones=5, calentamiento=2)
        diferencia = (y_cpu - y_mps.cpu()).abs().max().item()
        print(f"MPS/Metal: {tiempo_mps:.4f} segundos por convolución")
        print(f"Aceleración CPU/MPS: {tiempo_cpu / tiempo_mps:.2f}x")
        print(f"Diferencia máxima CPU vs MPS: {diferencia:.6f}")
    else:
        print("Tampoco está disponible MPS/Metal; solo se midió CPU.")


Entrada: (8, 32, 128, 128)
Formato: (n_batch, channels, height, width)
Kernel:  (64, 32, 3, 3)
Formato del kernel: (channels_out, channels_in, kernel_height, kernel_width)
Salida esperada: (8, 64, 128, 128)
CPU:  0.1194 segundos por convolución
CUDA no está disponible en este equipo.
Tampoco está disponible MPS/Metal; solo se midió CPU.
